# Classical Baseline: U-Net + ResNet50

**ISIC 2018 Skin Lesion Segmentation**

This notebook trains a U-Net with a ResNet50 backbone as a classical baseline for comparison against our proposed Quantum SAM architecture.

## Experimental Setup

All hyperparameters and data splits match the Quantum SAM training setup to enable a fair comparison:

| Parameter | Value |
|---|---|
| Dataset | ISIC 2018 Task 1 |
| Train / Val / Test split | 70% / 15% / 15% (seed=42) |
| Loss | Focal Tversky + BCE (combined) |
| Augmentation | Albumentations (flip, rotate, color jitter) |
| Optimizer | AdamW with cosine schedule |
| Epochs | 80 with early stopping |

The only difference from the Quantum SAM run is the model architecture itself.

## 1. Setup

In [ ]:
# 1.1 - Verify GPU availability
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# 1.2 - Install dependencies
!pip install segmentation_models_pytorch albumentations -q

In [ ]:
# 1.3 - Configure Kaggle API
import os
from google.colab import files
print("Upload your kaggle.json:")
files.upload()
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

!pip uninstall kaggle kagglesdk -y -q
!pip install kaggle -q

In [ ]:
# 1.4 - Download ISIC 2018 dataset
for d in ['data/raw', 'models/best', 'outputs/visualizations']:
    os.makedirs(d, exist_ok=True)

!kaggle datasets download -d tschandl/isic2018-challenge-task1-data-segmentation -p data/raw/ --unzip
print("ISIC 2018 dataset ready.")

## 2. Configuration and Imports

In [ ]:
# 2.1 - Imports
import os, cv2, random, math, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

import warnings
warnings.filterwarnings('ignore')

# Match the Quantum SAM seed for reproducible comparison
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda')
print(f"Device: {DEVICE}")

In [ ]:
# 2.2 - Configuration
CONFIG = {
    'seg_images': 'data/raw/ISIC2018_Task1-2_Training_Input',
    'seg_masks':  'data/raw/ISIC2018_Task1_Training_GroundTruth',

    # Model architecture
    'image_size':     384,
    'encoder_name':   'resnet50',
    'encoder_weights': 'imagenet',

    # Training
    'batch_size':      8,
    'epochs':          80,
    'lr':              1e-4,
    'weight_decay':    1e-4,
    'warmup_epochs':   5,
    'early_stopping':  15,

    # Loss (matched to Quantum SAM)
    'focal_gamma':    0.75,
    'tversky_alpha':  0.3,
    'tversky_beta':   0.7,
}

print("U-Net Baseline Configuration")
print(f"  Backbone: {CONFIG['encoder_name']} (pretrained on {CONFIG['encoder_weights']})")
print(f"  Image size: {CONFIG['image_size']}px")
print(f"  Epochs: {CONFIG['epochs']}, LR: {CONFIG['lr']}")

## 3. Dataset

The dataset split uses the same seed (42) and ratios (70/15/15) as the Quantum SAM run, ensuring identical train/val/test files across both experiments.

In [ ]:
# 3.1 - Standard segmentation dataset
class SkinSegDataset(Dataset):
    """
    Standard segmentation dataset for U-Net.
    Returns (image, mask) tuples — no bounding box prompts.
    """
    def __init__(self, image_dir, mask_dir, file_list, image_size=384, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.files = file_list
        self.image_size = image_size
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img_path = os.path.join(self.image_dir, fname)
        mask_path = os.path.join(self.mask_dir, fname.replace('.jpg', '_segmentation.png'))

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.float32)

        # Resize
        image = cv2.resize(image, (self.image_size, self.image_size))
        mask = cv2.resize(mask, (self.image_size, self.image_size),
                          interpolation=cv2.INTER_NEAREST)

        # Augmentation
        if self.transform:
            sample = self.transform(image=image, mask=mask)
            image = sample['image']
            mask = sample['mask']
        else:
            image = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0
            mask = torch.from_numpy(mask).float()

        # ImageNet normalization for ResNet50
        if isinstance(image, torch.Tensor) and image.max() > 1.5:
            image = image / 255.0
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        image = (image - mean) / std

        return image, mask.unsqueeze(0)

In [ ]:
# 3.2 - Augmentation pipeline (matched to Quantum SAM)
def get_transforms(mode='train'):
    if mode == 'train':
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.ShiftScaleRotate(
                shift_limit=0.1, scale_limit=0.15,
                rotate_limit=30, p=0.5
            ),
            A.RandomBrightnessContrast(p=0.3),
            A.HueSaturationValue(p=0.3),
            ToTensorV2(),
        ])
    return A.Compose([ToTensorV2()])

In [ ]:
# 3.3 - Train/Val/Test split (identical to Quantum SAM)
def split_dataset(image_dir, train_ratio=0.70, val_ratio=0.15, seed=42):
    random.seed(seed)
    images = sorted([f for f in os.listdir(image_dir) if f.endswith('.jpg')])
    random.shuffle(images)
    n = len(images)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    return (images[:n_train],
            images[n_train:n_train+n_val],
            images[n_train+n_val:])

train_files, val_files, test_files = split_dataset(CONFIG['seg_images'])
print(f"Train: {len(train_files)} | Val: {len(val_files)} | Test: {len(test_files)}")

## 4. Model

U-Net with a ResNet50 encoder pretrained on ImageNet, instantiated via `segmentation_models_pytorch`. This is a standard medical image segmentation baseline widely used in the literature.

In [ ]:
# 4.1 - Build model
model = smp.Unet(
    encoder_name=CONFIG['encoder_name'],
    encoder_weights=CONFIG['encoder_weights'],
    in_channels=3,
    classes=1,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"U-Net + {CONFIG['encoder_name']}")
print(f"  Total params:     {n_params:,}")
print(f"  Trainable params: {n_trainable:,}")

In [ ]:
# 4.2 - Loss function (identical to Quantum SAM)
class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, gamma=0.75):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma

    def forward(self, pred, target):
        pred = torch.sigmoid(pred)
        smooth = 1e-6
        tp = (pred * target).sum(dim=(1, 2, 3))
        fp = ((1 - target) * pred).sum(dim=(1, 2, 3))
        fn = (target * (1 - pred)).sum(dim=(1, 2, 3))
        tversky = (tp + smooth) / (tp + self.alpha * fp + self.beta * fn + smooth)
        focal_tversky = (1 - tversky) ** self.gamma
        return focal_tversky.mean()

class CombinedLoss(nn.Module):
    """Focal Tversky + BCE — matched to Quantum SAM training."""
    def __init__(self, alpha=0.3, beta=0.7, gamma=0.75):
        super().__init__()
        self.ft = FocalTverskyLoss(alpha, beta, gamma)
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, pred, target):
        return 0.7 * self.ft(pred, target) + 0.3 * self.bce(pred, target)

## 5. Training

In [ ]:
# 5.1 - Training and evaluation functions
def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss = 0
    for images, masks in tqdm(loader, desc='  Train', leave=False):
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            pred = model(images)
            loss = criterion(pred, masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, criterion, threshold=0.5):
    model.eval()
    total_loss = 0
    all_ious, all_dices = [], []
    with torch.no_grad():
        for images, masks in tqdm(loader, desc='  Val', leave=False):
            images = images.to(DEVICE)
            masks = masks.to(DEVICE)
            pred = model(images)
            loss = criterion(pred, masks)
            total_loss += loss.item()

            pred_bin = (torch.sigmoid(pred) > threshold).float()
            inter = (pred_bin * masks).sum(dim=(1, 2, 3))
            union = (pred_bin + masks).clamp(0, 1).sum(dim=(1, 2, 3))
            iou = (inter + 1e-6) / (union + 1e-6)
            dice = (2 * inter + 1e-6) / (
                pred_bin.sum(dim=(1,2,3)) + masks.sum(dim=(1,2,3)) + 1e-6
            )
            all_ious.extend(iou.cpu().tolist())
            all_dices.extend(dice.cpu().tolist())
    return total_loss / len(loader), np.mean(all_ious), np.mean(all_dices)

In [ ]:
# 5.2 - DataLoaders
train_ds = SkinSegDataset(
    CONFIG['seg_images'], CONFIG['seg_masks'], train_files,
    image_size=CONFIG['image_size'],
    transform=get_transforms('train')
)
val_ds = SkinSegDataset(
    CONFIG['seg_images'], CONFIG['seg_masks'], val_files,
    image_size=CONFIG['image_size'],
    transform=None
)

train_loader = DataLoader(
    train_ds, batch_size=CONFIG['batch_size'], shuffle=True,
    num_workers=4, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_ds, batch_size=CONFIG['batch_size'], shuffle=False,
    num_workers=4, pin_memory=True
)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
# 5.3 - Training loop
criterion = CombinedLoss(
    alpha=CONFIG['tversky_alpha'],
    beta=CONFIG['tversky_beta'],
    gamma=CONFIG['focal_gamma']
)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=CONFIG['lr'],
    weight_decay=CONFIG['weight_decay']
)
scaler = torch.cuda.amp.GradScaler()

def lr_lambda(epoch):
    if epoch < CONFIG['warmup_epochs']:
        return (epoch + 1) / CONFIG['warmup_epochs']
    progress = (epoch - CONFIG['warmup_epochs']) / max(
        1, CONFIG['epochs'] - CONFIG['warmup_epochs']
    )
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

history = {'train_loss': [], 'val_loss': [], 'val_iou': [], 'val_dice': [], 'lr': []}
best_iou = 0
patience = 0

print("Starting U-Net baseline training")
print("=" * 70)

for epoch in range(CONFIG['epochs']):
    current_lr = optimizer.param_groups[0]['lr']
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
    val_loss, val_iou, val_dice = evaluate(model, val_loader, criterion)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_iou'].append(val_iou)
    history['val_dice'].append(val_dice)
    history['lr'].append(current_lr)

    msg = (
        f"  E{epoch+1:02d}/{CONFIG['epochs']} | "
        f"T:{train_loss:.4f} V:{val_loss:.4f} | "
        f"IoU:{val_iou:.4f} D:{val_dice:.4f} | "
        f"LR:{current_lr:.6f}"
    )

    if val_iou > best_iou:
        best_iou = val_iou
        patience = 0
        torch.save({
            'model_state': model.state_dict(),
            'best_iou': best_iou,
            'epoch': epoch,
        }, 'models/best/best_unet_baseline.pth')
        print(f"{msg}  [best]")
    else:
        patience += 1
        print(f"{msg}  ({patience}/{CONFIG['early_stopping']})")
        if patience >= CONFIG['early_stopping']:
            print(f"  Early stopping triggered at epoch {epoch+1}")
            break

print(f"\n{'=' * 70}")
print(f"U-Net baseline training complete. Best validation IoU: {best_iou:.4f}")

## 6. Test Set Evaluation

In [ ]:
# 6.1 - Load best model and evaluate on test set
ckpt = torch.load('models/best/best_unet_baseline.pth')
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f"Loaded best model — val IoU: {ckpt['best_iou']:.4f} at epoch {ckpt['epoch']}")

test_ds = SkinSegDataset(
    CONFIG['seg_images'], CONFIG['seg_masks'], test_files,
    image_size=CONFIG['image_size'], transform=None
)
test_loader = DataLoader(
    test_ds, batch_size=CONFIG['batch_size'], shuffle=False,
    num_workers=4, pin_memory=True
)

all_ious, all_dices = [], []
with torch.no_grad():
    for images, masks in tqdm(test_loader, desc='Test'):
        images = images.to(DEVICE)
        pred = model(images)
        pred = torch.sigmoid(pred).cpu()
        pred_bin = (pred > 0.5).float()

        inter = (pred_bin * masks).sum(dim=(1, 2, 3))
        union = (pred_bin + masks).clamp(0, 1).sum(dim=(1, 2, 3))
        iou = (inter + 1e-6) / (union + 1e-6)
        dice = (2 * inter + 1e-6) / (pred_bin.sum(dim=(1,2,3)) + masks.sum(dim=(1,2,3)) + 1e-6)
        all_ious.extend(iou.tolist())
        all_dices.extend(dice.tolist())

print(f"\n{'=' * 60}")
print(f"U-NET BASELINE TEST RESULTS")
print(f"{'=' * 60}")
print(f"  IoU:         {np.mean(all_ious):.4f} (±{np.std(all_ious):.4f})")
print(f"  Dice:        {np.mean(all_dices):.4f}")
print(f"  IoU min/max: {np.min(all_ious):.4f} / {np.max(all_ious):.4f}")
print(f"{'=' * 60}")

In [ ]:
# 6.2 - Persist results for cross-model comparison
import json
results = {
    'model': 'U-Net + ResNet50',
    'best_val_iou': float(best_iou),
    'test_iou_mean': float(np.mean(all_ious)),
    'test_iou_std': float(np.std(all_ious)),
    'test_dice': float(np.mean(all_dices)),
    'config': {
        'image_size': CONFIG['image_size'],
        'epochs': CONFIG['epochs'],
        'lr': CONFIG['lr'],
        'seed': SEED,
    }
}
with open('outputs/unet_baseline_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("Results saved to outputs/unet_baseline_results.json")

## 7. Visualization

In [ ]:
# 7.1 - Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
er = range(1, len(history['train_loss']) + 1)

axes[0].plot(er, history['train_loss'], 'b-', lw=2, label='Train')
axes[0].plot(er, history['val_loss'], 'r-', lw=2, label='Validation')
axes[0].set_title('Loss', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(er, history['val_iou'], 'g-', lw=2, label='Val IoU')
axes[1].plot(er, history['val_dice'], 'm-', lw=2, label='Val Dice')
axes[1].set_title('Validation Metrics', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/visualizations/unet_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 7.2 - Sample predictions on test set
fig, axes = plt.subplots(4, 3, figsize=(15, 20))
indices = np.random.choice(len(test_ds), 4, replace=False)

for row, idx in enumerate(indices):
    img, mask = test_ds[idx]
    with torch.no_grad():
        pred = model(img.unsqueeze(0).to(DEVICE))
        pred = torch.sigmoid(pred).cpu().squeeze().numpy()
        pred_bin = (pred > 0.5).astype(np.float32)

    img_vis = img.numpy().transpose(1, 2, 0)
    img_vis = (img_vis * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))
    img_vis = np.clip(img_vis, 0, 1)

    axes[row, 0].imshow(img_vis)
    axes[row, 0].set_title('Input image')
    axes[row, 0].axis('off')
    axes[row, 1].imshow(mask.squeeze(), cmap='gray')
    axes[row, 1].set_title('Ground truth')
    axes[row, 1].axis('off')
    axes[row, 2].imshow(pred_bin, cmap='gray')
    axes[row, 2].set_title('U-Net prediction')
    axes[row, 2].axis('off')

plt.tight_layout()
plt.savefig('outputs/visualizations/unet_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Artifacts to Drive

In [ ]:
import shutil, glob
from google.colab import drive
drive.mount('/content/drive')

save_dir = '/content/drive/MyDrive/skin_cancer_project/baselines'
os.makedirs(save_dir, exist_ok=True)

shutil.copy('models/best/best_unet_baseline.pth', save_dir)
shutil.copy('outputs/unet_baseline_results.json', save_dir)
for f in glob.glob('outputs/visualizations/unet_*.png'):
    shutil.copy(f, save_dir)

print(f"U-Net baseline artifacts saved to: {save_dir}")